# Data Cleaning & Feature Engineering Notebook

#### This notebook contains code to perform following data cleaning on raw data files

- Handling Missing Values  
- Dropping duplicate records 
- Extracting JSON data into meaningful columns  
- Merging data files to obtain two cleaned datasets for further analysis. 

#### The notebook also contains code for following Feature Engineering on the cleaned datasets
   
- Budget Tier - Low, Mid, High, and Blockbuster (Quartile Based)
  
  Budget tiers helps in quantifying the impact of budget on financial success of a movie
- Studio Tier - Major Studio or Independent
  
  Studio tiers helps in investigating the impact of studio size on audience reception of a movie
- Release Season – Categorical column, Summer (June, July, August), Year End (November, December) or other
  
  Release season helps in studying the effect of release timing on Financial success of a moive.
- ROI - Calculated as (revenue - budget) / budget
  
  ROI helps in analyzing which movies generate the highest profits and which ones make losses.
- Weighted Rating – Calculated as (v/(v+m)) * R + (m/(v+m)) * C,

Where
        R = movie average rating  
        v = review count  
        C = overall average rating across dataset  
        m = minimum review threshold 

Weighted Rating formula makes the rating more balanced by ensuring- 
1. movies with very low reviews get pulled toward the global average
2. highly reviewed movies retain their actual rating.

### Importing the Libraries

In [87]:
import pandas as pd
import numpy as np
import ast # for extracting JSON columns

### Importing Datasets into DataFrames

In [3]:
movies_df = pd.read_csv('movies_metadata.csv', engine='python') # using python engine for parsing without errors
ratings_df = pd.read_csv('ratings.csv') 
credits_df =pd.read_csv('credits.csv',engine='python')
links_df = pd.read_csv('links.csv')

### Examining the ratings Dataframe

In [4]:
ratings_df.shape

(26024289, 4)

In [5]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [6]:
ratings_df.isna().sum() # Checking total missing values in each column

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

### Aggregating the ratings dataframe to get 1 row per movie

In [7]:
ratings_agg_df = ratings_df.groupby('movieId').agg({'rating':'mean','userId':'count'})

In [8]:
ratings_agg_df.head()

,rating,userId
movieId,,
1,3.888157,66008
2,3.236953,26060
3,3.175550,15497
4,2.875713,2981
5,3.079565,15258


In [9]:
ratings_agg_df.shape

(45115, 2)

### Cleaning the aggregated ratings dataframe

In [10]:
ratings_agg_df.reset_index(inplace=True)
ratings_agg_df.rename(columns = {'rating':'avg_rating','userId':'review_count'},inplace=True)
ratings_agg_df['avg_rating'] = ratings_agg_df['avg_rating'].round(2)


In [12]:
ratings_agg_df.dtypes # confirming the datatype of each column

movieId           int64
avg_rating      float64
review_count      int64
dtype: object

### Examining the movies dataframe

In [13]:
movies_df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [14]:
movies_df.shape

(45466, 24)

In [15]:
movies_df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

### Cleaning the movies dataframe

#### Keeping only those columns needed in Analysis

In [16]:
movies_df = movies_df[['budget','genres','id','popularity','production_companies','release_date','revenue','runtime','title']]

In [17]:
movies_df.dtypes # Examining if all columns have correct datatype

budget                   object
genres                   object
id                       object
popularity               object
production_companies     object
release_date             object
revenue                 float64
runtime                 float64
title                    object
dtype: object

#### Correcting Datatype for columns id, popularity,release_date and budget 

In [18]:
movies_df = movies_df[pd.to_numeric(movies_df['id'], errors='coerce').notnull()] # Removing missing ids
movies_df['id'] = movies_df['id'].astype(int)
movies_df['budget'] = pd.to_numeric(movies_df['budget'], errors='coerce')
movies_df['popularity'] = pd.to_numeric(movies_df['popularity'], errors='coerce')

In [19]:
movies_df['release_date'] = pd.to_datetime(movies_df['release_date'],errors='coerce')

In [20]:
movies_df.dtypes # confirming the datatypes are correct after transformation

budget                           int64
genres                          object
id                               int64
popularity                     float64
production_companies            object
release_date            datetime64[ns]
revenue                        float64
runtime                        float64
title                           object
dtype: object

#### Inspecting the columns genres and production_companies containing JSON Datastructures

In [21]:
movies_df['genres'][0]

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [22]:
movies_df['production_companies'][0]

"[{'name': 'Pixar Animation Studios', 'id': 3}]"

#### Extracting the primary genre and all genres from the genres column using custom functions

In [23]:
# Define function to extract primary(first in list) genre per movie
def extract_primary_genre(genre_str):
    try:
        genres = ast.literal_eval(genre_str)     # convert strings to python lists and dictionaries
        
        if isinstance(genres, list) and len(genres) > 0:
            return genres[0]['name']             # Extract the first name in list of genres
        
        return None
    
    except:
        return None
# Apply the function to genres column
movies_df['primary_genre'] = movies_df['genres'].apply(extract_primary_genre)

In [24]:
# Define function to extract all genres per movie
def extract_all_genres(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        
        if isinstance(genres, list) and len(genres) > 0:
            return ' | '.join(
                genre['name'] for genre in genres    # generator expression to get all genres and join to concatenate using pipe
            )
        
        return None
    
    except:
        return None

#Apply the function to genres column
movies_df['all_genres'] = movies_df['genres'].apply(extract_all_genres)

#### Extracting the primary production company from the production_companies column using custom function

In [26]:
# Define function to extract primary (first in list) production company per movie
def extract_first_company(company_str):
    try:
        companies = ast.literal_eval(company_str)
        
        if isinstance(companies, list) and len(companies) > 0:
            return companies[0]['name']
        
        return None
    
    except:
        return None

#Apply the function to genres column
movies_df['primary_company'] = movies_df['production_companies'].apply(extract_first_company)

#### Dropping the genres and production companies columns from the movies dataframe

In [27]:
movies_df.drop(columns=['production_companies','genres'], inplace=True)

In [28]:
movies_df.head() # Inspecting the updated dataframe

,budget,id,popularity,release_date,revenue,runtime,title,primary_genre,all_genres,primary_company
0,30000000,862,21.946943,1995-10-30,373554033.0,81.0,Toy Story,Animation,Animation | Comedy | Family,Pixar Animation Studios
1,65000000,8844,17.015539,1995-12-15,262797249.0,104.0,Jumanji,Adventure,Adventure | Fantasy | Family,TriStar Pictures
2,0,15602,11.712900,1995-12-22,0.0,101.0,Grumpier Old Men,Romance,Romance | Comedy,Warner Bros.
3,16000000,31357,3.859495,1995-12-22,81452156.0,127.0,Waiting to Exhale,Comedy,Comedy | Drama | Romance,Twentieth Century Fox Film Corporation
4,0,11862,8.387519,1995-02-10,76578911.0,106.0,Father of the Bride Part II,Comedy,Comedy,Sandollar Productions


#### Handling missing values in movies dataframe

In [29]:
movies_df.isna().sum()

budget                 0
id                     0
popularity             3
release_date          87
revenue                3
runtime              260
title                  3
primary_genre       2442
all_genres          2442
primary_company    11878
dtype: int64

In [30]:
movies_df.dropna(              # Dropping rows with missing values in critical columns
    subset=[
        'popularity',
        'revenue',
        'title',
        'release_date',      
        'primary_genre',
        'all_genres'
    ],
    inplace=True
)

In [31]:
movies_df['runtime'].fillna(     # Imputing median runtime for missing values in runtime column            
    movies_df['runtime'].median(),
    inplace=True
)

/tmp/ipykernel_2078/466434147.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movies_df['runtime'].fillna(     # Imputing median runtime for missing values in runtime column


In [32]:
movies_df.isna().sum() # Verifying missing values after above cleaning.
                       # Missing values in primary_company are kept as it is not the most important feature.

budget                0
id                    0
popularity            0
release_date          0
revenue               0
runtime               0
title                 0
primary_genre         0
all_genres            0
primary_company    9756
dtype: int64

In [33]:
movies_df['popularity'] = movies_df['popularity'].round(2) # Rounding the popularity to 2 decimal places

### Cleaning the credits dataframe and extracting director from nested JSON 

In [34]:
credits_df.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [35]:
credits_df['crew'][1]  # Examining the nested structure to extract director name

"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'department': 'Production', 'gender': 2, 'id': 511, 'job': 'Executive Producer', 'name': 'Larry J. Franco', 'profile_path': None}, {'credit_id': '52fe44bfc3a36847f80a7c89', 'department': 'Writing', 'gender': 2, 'id': 876, 'job': 'Screenplay', 'name': 'Jonathan Hensleigh', 'profile_path': '/l1c4UFD3g0HVWj5f0CxXAvMAGiT.jpg'}, {'credit_id': '52fe44bfc3a36847f80a7cdd', 'department': 'Sound', 'gender': 2, 'id': 1729, 'job': 'Original Music Composer', 'name': 'James Horner', 'profile_path': '/oLOtXxXsYk8X4qq0ud4xVypXudi.jpg'}, {'credit_id': '52fe44bfc3a36847f80a7c7d', 'department': 'Directing', 'gender': 2, 'id': 4945, 'job': 'Director', 'name': 'Joe Johnston', 'profile_path': '/fok4jaO62v5IP6hkpaaAcXuw2H.jpg'}, {'credit_id': '52fe44bfc3a36847f80a7cd7', 'department': 'Editing', 'gender': 2, 'id': 4951, 'job': 'Editor', 'name': 'Robert Dalva', 'profile_path': None}, {'credit_id': '573523bec3a368025100062c', 'department': 'Production', 'gender': 0, '

In [36]:
credits_df = credits_df[['id', 'crew']] # Keeping only relevant columns in the dataframe

#### Extracting the Director of each movie from JSON structure using custom function

In [37]:
#Define function to extract director's name for each movie 
def extract_director(crew_str):
    try:
        crew_list = ast.literal_eval(crew_str)
        
        if isinstance(crew_list, list):
            
            for person in crew_list:
                
                if person.get('job') == 'Director':
                    return person.get('name')
        
        return None
    
    except:
        return None

# Apply the function to crew column         
credits_df['director'] = credits_df['crew'].apply(extract_director)


In [38]:
credits_df.head()

,id,crew,director
0,862,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",John Lasseter
1,8844,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",Joe Johnston
2,15602,"[{'credit_id': '52fe466a9251416c75077a89', 'de...",Howard Deutch
3,31357,"[{'credit_id': '52fe44779251416c91011acb', 'de...",Forest Whitaker
4,11862,"[{'credit_id': '52fe44959251416c75039ed7', 'de...",Charles Shyer


In [39]:
credits_df.drop(columns =['crew'],inplace=True) # Dropping the crew column from dataframe 

#### Handling missing values in credits dataframe

In [40]:
credits_df.isna().sum()

id            0
director    887
dtype: int64

In [41]:
credits_df.dropna(subset=['director'], inplace=True) # Dropping missing values from Director Column

In [42]:
credits_df.shape

(44589, 2)

In [43]:
credits_df.dtypes # Validating Datatype of each column

id           int64
director    object
dtype: object

### Cleaning the links dataframe to prepare for merging dataframes

In [44]:
links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [45]:
links_df.shape

(45843, 3)

In [46]:
links_df.dtypes

movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

In [47]:
links_df.isna().sum()

movieId      0
imdbId       0
tmdbId     219
dtype: int64

In [48]:
links_df.dropna(subset=['tmdbId'], inplace=True) # Dropping the rows with missing tmdbIds

In [49]:
links_df['tmdbId'] = links_df['tmdbId'].astype(int) # Converting tmdbId to Integers

### Merging the Dataframes to cretae final cleaned dataset

#### Create mapping between movieId and tmdbId by merging ratings with links

In [50]:
ratings_with_links = ratings_agg_df.merge(
                     links_df,
                     on = 'movieId',
                     how ='inner')

In [51]:
ratings_with_links.shape

(44902, 5)

#### Create consolidated dataframe by merging resulting dataframe with movies dataframe

In [52]:
final_df = movies_df.merge(
    ratings_with_links,
    left_on='id',
    right_on='tmdbId',
    how='inner'
)

In [53]:
final_df.shape

(42391, 15)

In [54]:
final_df['release_year'] = final_df['release_date'].dt.year # Create feature for release year

#### Add Director information by merging the result with credits dataframe

In [55]:
credits_df.head()

,id,director
0,862,John Lasseter
1,8844,Joe Johnston
2,15602,Howard Deutch
3,31357,Forest Whitaker
4,11862,Charles Shyer


In [57]:
final_df =final_df.merge( 
          credits_df,
          on ='id',
          how ='left')

In [58]:
final_df.shape

(42534, 17)

#### Checking missing values in the final dataframe

In [59]:
final_df.isna().sum()

budget                0
id                    0
popularity            0
release_date          0
revenue               0
runtime               0
title                 0
primary_genre         0
all_genres            0
primary_company    9668
movieId               0
avg_rating            0
review_count          0
imdbId                0
tmdbId                0
release_year          0
director            539
dtype: int64

In [60]:
final_df['director'].fillna(     #Imputing 'Unknown Director for missing values
    'Unknown Director',
    inplace=True
)

/tmp/ipykernel_2078/1388308552.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['director'].fillna(     #Imputing 'Unknown Director for missing values


In [61]:
final_df.columns

Index(['budget', 'id', 'popularity', 'release_date', 'revenue', 'runtime',
       'title', 'primary_genre', 'all_genres', 'primary_company', 'movieId',
       'avg_rating', 'review_count', 'imdbId', 'tmdbId', 'release_year',
       'director'],
      dtype='object')

#### Dropping columns id,tmdbId and imdbId as they are not required in analysis

In [62]:
final_df.drop(                             
    columns=['id', 'tmdbId', 'imdbId'],
    inplace=True
)

#### Rearranging columns in final dataframe to improve readability

In [64]:
final_df = final_df[
    [
        'movieId',
        'title',
        'release_date',
        'release_year',
        'budget',
        'popularity',
        'revenue',
        'runtime',
        'primary_company',
        'primary_genre',
        'all_genres',
        'avg_rating',
        'review_count',
        'director'
    ]
]

In [65]:
final_df.head()

,movieId,title,release_date,release_year,budget,popularity,revenue,runtime,primary_company,primary_genre,all_genres,avg_rating,review_count,director
0,1,Toy Story,1995-10-30,1995,30000000,21.95,373554033.0,81.0,Pixar Animation Studios,Animation,Animation | Comedy | Family,3.89,66008,John Lasseter
1,2,Jumanji,1995-12-15,1995,65000000,17.02,262797249.0,104.0,TriStar Pictures,Adventure,Adventure | Fantasy | Family,3.24,26060,Joe Johnston
2,3,Grumpier Old Men,1995-12-22,1995,0,11.71,0.0,101.0,Warner Bros.,Romance,Romance | Comedy,3.18,15497,Howard Deutch
3,4,Waiting to Exhale,1995-12-22,1995,16000000,3.86,81452156.0,127.0,Twentieth Century Fox Film Corporation,Comedy,Comedy | Drama | Romance,2.88,2981,Forest Whitaker
4,5,Father of the Bride Part II,1995-02-10,1995,0,8.39,76578911.0,106.0,Sandollar Productions,Comedy,Comedy,3.08,15258,Charles Shyer


#### Checking for count of non-zero values in budget and revenue columns

In [66]:
final_df[(final_df['revenue'] != 0) & (final_df['budget'] != 0) ].shape

(5401, 14)

### Creating csv files from the dataframe further analysis 

In [67]:
financial_df = final_df[             # Creating separate financial dataset with non-zero values for budget and revenue.
    (final_df['budget'] > 0) &       # This dataset will be used for Regression Analysis 
    (final_df['revenue'] > 0)
].copy() 

### Dropping duplicate movieIds from financial dataframe and final dataframe

In [70]:
financial_df[financial_df.duplicated(subset='movieId', keep=False)].shape # Checking duplicate movieIds in financial dataframe

(48, 14)

In [71]:
# Obtaining the movieIds with highest duplicate records
financial_df.groupby('movieId').agg({'title':'count'}).sort_values(by='title',ascending =False).head()

,title
movieId,
112062,4
976,4
174533,4
77141,4
47237,4


In [73]:
financial_df[financial_df['movieId'] == 112062].T  # Inspecting one of the movieIds to confirm the rows are identical

,4276,4277,22927,22928
movieId,112062,112062,112062,112062
title,Camille Claudel 1915,Camille Claudel 1915,Camille Claudel 1915,Camille Claudel 1915
release_date,2013-03-13 00:00:00,2013-03-13 00:00:00,2013-03-13 00:00:00,2013-03-13 00:00:00
release_year,2013,2013,2013,2013
budget,3512454,3512454,3512454,3512454
popularity,0.13,0.13,0.11,0.11
revenue,115860.0,115860.0,115860.0,115860.0
runtime,95.0,95.0,95.0,95.0
primary_company,Canal+,Canal+,Canal+,Canal+
primary_genre,Drama,Drama,Drama,Drama


### Observation
Since the rows are identical except for the popularity which is 0.13 for 2 rows and 0.11 for 2 rows. The best strategy would be to keep one row with highest popularity and drop the rest.

In [74]:
financial_df = (
    financial_df
    .sort_values('popularity', ascending=False)
    .drop_duplicates(subset='movieId', keep='first')
)

In [76]:
financial_df['movieId'].duplicated().sum() # Verifying that no duplicate record remains

0

In [77]:
financial_df = (          #sorting again by movieId to aid readability and resetting index to obtain consistent indexes
    financial_df
    .sort_values('movieId')
    .reset_index(drop=True)
)

In [78]:
financial_df.shape  # Verifying the final shape of financial dataframe

(5365, 14)

In [79]:
final_df['movieId'].duplicated().sum() # Checking duplicates in final dataframe

205

In [81]:
# Obtaining the movieIds with highest duplicate records
final_df.groupby('movieId').agg({'title':'count'}).sort_values(by='title',ascending =False)

,title
movieId,
65243,9
85070,9
66140,9
98460,4
123107,4
...,...
72124,1
72129,1
72131,1


In [83]:
final_df[final_df['movieId'] == 65243].T # Inspecting one of the movieIds to confirm the rows are identical

,13125,13126,13127,13243,13244,13245,16548,16549,16550
movieId,65243,65243,65243,65243,65243,65243,65243,65243,65243
title,Blackout,Blackout,Blackout,Blackout,Blackout,Blackout,Blackout,Blackout,Blackout
release_date,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00,2008-12-26 00:00:00
release_year,2008,2008,2008,2008,2008,2008,2008,2008,2008
budget,0,0,0,0,0,0,0,0,0
popularity,0.41,0.41,0.41,0.41,0.41,0.41,0.41,0.41,0.41
revenue,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
runtime,108.0,108.0,108.0,108.0,108.0,108.0,108.0,108.0,108.0
primary_company,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine,Filmiteollisuus Fine
primary_genre,Thriller,Thriller,Thriller,Thriller,Thriller,Thriller,Thriller,Thriller,Thriller


### Observation
Since the rows are identical in all the columns the best strategy is to keep one row per movieId and drop the rest.

In [84]:
final_df =final_df.drop_duplicates(subset = 'movieId',keep ='first')

In [85]:
final_df['movieId'].duplicated().sum() # Verifying that no duplicate record remains

0

In [86]:
final_df.shape # Verifying the final shape of final dataframe

(42329, 14)

## Feature Engineering 
Since there were many zero values in budget and revenue columns in the final dataframe,Features which require consistent financial data such as ROI,budget tier and studio tier have been added to the financial dataset only. The other features like Weighted Rating and Release season which do not depened on financial data have been added to both datasets.

#### Adding Budget Tier Feature to Financial Dataframe

In [88]:
financial_df['budget_tier'] = pd.qcut(
    financial_df['budget'],
    q=4,
    labels=['Low', 'Mid', 'High', 'Blockbuster']
)

In [90]:
financial_df['budget_tier'].value_counts() #Verifying the sizes of each tier

budget_tier
High           1392
Mid            1380
Low            1343
Blockbuster    1250
Name: count, dtype: int64

In [92]:
# Adding a log budget column to aid in statistical analysis
financial_df['log_budget'] = np.log1p(financial_df['budget']).round(2)

#### Adding Studio Tier Feature to Financial Dataframe

In [96]:
# Creating a log revenue column to help later in statistical analysis
financial_df['log_revenue'] = np.log1p(financial_df['revenue']).round(2)

In [97]:
# Creating a grouped dataframe with statistics for each studio
studio_df = (
    financial_df
    .groupby('primary_company')
    .agg(
        movie_count=('movieId', 'count'),
        total_revenue=('revenue', 'sum'),
        median_revenue=('revenue', 'median'),
        avg_log_revenue=('log_revenue', 'mean')
    ).sort_values(by=['movie_count'],ascending =[False])
)


In [99]:
#Inspecting the statistics for studios credited for more than 5 movies
studio_df[studio_df['movie_count'] >= 5].describe()

,movie_count,total_revenue,median_revenue,avg_log_revenue
count,131.000000,1.310000e+02,1.310000e+02,131.000000
mean,26.664122,3.174706e+09,6.780752e+07,16.906684
std,52.584966,7.569166e+09,1.122116e+08,1.386834
min,5.000000,2.689868e+06,6.242095e+05,12.710000
25%,7.000000,1.973204e+08,1.218691e+07,15.897000
50%,9.000000,6.959230e+08,3.529447e+07,17.056061
75%,17.500000,2.374834e+09,7.088112e+07,17.835925
max,336.000000,4.531844e+10,7.106842e+08,20.420000


### Observation
Studios producing at least five films within the financial dataset were classified as Major Studios. This threshold represented roughly the top 10% of studios by production volume and also aligned with substantially higher aggregate revenues. 

In [100]:
studio_df['studio_tier'] = np.where(studio_df['movie_count'] >= 5,1,0) # Creating the classification of studios

In [101]:
#Also creating a studio label column to visualize studio stats in tableau dashboard 
studio_df['studio_label'] = np.where(studio_df['movie_count'] >= 5,'Major Studio','Independent')

In [106]:
# Merging the studio tier and studio label with Financial dataframe to add these features
financial_df = financial_df.merge(
    studio_df[['studio_tier','studio_label']],
    left_on='primary_company',
    right_index=True,
    how='left'
)

#### Adding Release season feature to financial dataframe

In [109]:
conditions = [
    financial_df['release_date'].dt.month.isin([6, 7, 8]),
    financial_df['release_date'].dt.month.isin([11, 12])
]

choices = ['Summer', 'Year-End']

financial_df['release_season'] = np.select(
    conditions,
    choices,
    default='Other'
)

In [112]:
financial_df['release_season'].value_counts()   # Verifying the sizes of each category

release_season
Other       3035
Summer      1359
Year-End     971
Name: count, dtype: int64

In [113]:
financial_df.groupby('release_season')['revenue'].median() # Checking if release timing influences a movie's revenue

release_season
Other       22217407.0
Summer      38625550.0
Year-End    46442528.0
Name: revenue, dtype: float64

#### Observation
The size of each category is large enough for a statistical analysis later. Also the median revenue for the three categories differ by a significant amount. Whether these differences are statistically significant will be ascertained later by an ANOVA test.

#### Adding ROI Feature to financial dataframe

In [114]:
financial_df['roi'] = ((financial_df['revenue'] - financial_df['budget']) /financial_df['budget']).round(2)

In [116]:
# Checking the distribution of the roi column
financial_df['roi'].quantile([0.01,0.05,0.95,0.99])

0.01    -1.0000
0.05    -0.9300
0.95    14.5940
0.99    83.2232
Name: roi, dtype: float64

#### Observation 

ROI distribution is extremely right-skewed with many outliers 

| Quantile    | Meaning                               |
| ----------- | ------------------------------------- |
| 1% = -1.00  | Total financial loss                  |
| 5% = -0.93  | Severe losses are common              |
| 95% = 14.59 | Top films earn ~14x budget profit     |
| 99% = 83.22 | Extreme outlier films earn 83x budget |

These outliers will be problematic if roi is used later in regression/correlation analysis but dropping them from dataset would delete important movies. Thus adding another column which caps the upper and lower roi's seems the best startegy. This will help reduce distortion while 
maintain dataset size.

In [118]:
lower = financial_df['roi'].quantile(0.01)
upper = financial_df['roi'].quantile(0.99)

financial_df['roi_capped'] = financial_df['roi'].clip(lower, upper) # Creating roi_capped column

In [119]:
financial_df.head()

,movieId,title,release_date,release_year,budget,popularity,revenue,runtime,primary_company,primary_genre,...,review_count,director,budget_tier,log_budget,log_revenue,studio_tier,studio_label,release_season,roi,roi_capped
0,1,Toy Story,1995-10-30,1995,30000000,21.95,373554033.0,81.0,Pixar Animation Studios,Animation,...,66008,John Lasseter,High,17.22,19.74,0.0,Independent,Other,11.45,11.45
1,2,Jumanji,1995-12-15,1995,65000000,17.02,262797249.0,104.0,TriStar Pictures,Adventure,...,26060,Joe Johnston,Blockbuster,17.99,19.39,1.0,Major Studio,Year-End,3.04,3.04
2,4,Waiting to Exhale,1995-12-22,1995,16000000,3.86,81452156.0,127.0,Twentieth Century Fox Film Corporation,Comedy,...,2981,Forest Whitaker,Mid,16.59,18.22,1.0,Major Studio,Year-End,4.09,4.09
3,6,Heat,1995-12-15,1995,60000000,17.92,187436818.0,170.0,Regency Enterprises,Action,...,27895,Michael Mann,Blockbuster,17.91,19.05,1.0,Major Studio,Year-End,2.12,2.12
4,9,Sudden Death,1995-12-22,1995,35000000,5.23,64350171.0,106.0,Universal Pictures,Action,...,4423,Peter Hyams,High,17.37,17.98,1.0,Major Studio,Year-End,0.84,0.84


#### Adding Weighted Rating column to financial dataframe

In [120]:
#Inspecting the distribution of avg_rating and review_count to help decide minimum review threshold
financial_df[['avg_rating','review_count']].describe()

,avg_rating,review_count
count,5365.000000,5365.000000
mean,3.201085,3822.267288
std,0.524608,7842.152593
min,0.500000,1.000000
25%,2.880000,182.000000
50%,3.250000,860.000000
75%,3.580000,3777.000000
max,5.000000,91921.000000


#### Observation
The minimum review threshold (m) was set equal to the median review count of the dataset to balance rating reliability 
while retaining sufficient variation across films.

In [121]:
min_review_threshold = financial_df['review_count'].median()

In [123]:
# Computing the overall average rating across the entire dataset
overall_avg_rating = (financial_df['avg_rating'] * financial_df['review_count']).sum()/financial_df['review_count'].sum()

In [124]:
overall_avg_rating

3.559436213381303

In [125]:
financial_df['weighted_rating'] = (
                             (financial_df['review_count']/(financial_df['review_count'] + min_review_threshold))*financial_df['avg_rating'] 
                              + 
                             (min_review_threshold/(financial_df['review_count'] + min_review_threshold)) * overall_avg_rating).round(2)

In [127]:
financial_df.shape   # Checking the final shape of dataframe after adding all features

(5365, 23)

#### Adding Release Season feature to final dataframe

In [129]:
conditions = [
    final_df['release_date'].dt.month.isin([6, 7, 8]),
    final_df['release_date'].dt.month.isin([11, 12])
]

choices = ['Summer', 'Year-End']

final_df['release_season'] = np.select(
    conditions,
    choices,
    default='Other'
)

In [130]:
final_df['release_season'].value_counts() # Checking size of each category

release_season
Other       26670
Summer       8670
Year-End     6989
Name: count, dtype: int64

#### Adding Weighted Rating feature to final dataframe

In [131]:
final_df['review_count'].describe() # Checking distribution of review count to decide minimum review threshold

count    42329.000000
mean       613.192941
std       3131.338884
min          1.000000
25%          2.000000
50%          9.000000
75%         81.000000
max      91921.000000
Name: review_count, dtype: float64

#### Observation
We can not use median as the minimum review threshold in this case as the distribution is highly skewed and median is just 9 reviews therefore the weighted formula would barely smooth anything. A 75% the percentile works well in this case as a movie will need substantially above-average review support before its own rating strongly dominates.

In [132]:
min_review_threshold_f = final_df['review_count'].quantile(0.75)

In [133]:
overall_avg_rating_f = (final_df['avg_rating'] * final_df['review_count']).sum()/final_df['review_count'].sum()

In [135]:
final_df['weighted_rating'] = (
                             (final_df['review_count']/(final_df['review_count'] + min_review_threshold_f))*final_df['avg_rating'] 
                              + 
                             (min_review_threshold_f/(final_df['review_count'] + min_review_threshold_f)) * overall_avg_rating_f).round(2)


In [139]:
final_df= final_df.reset_index(drop=True) # resetting index to keep consistent index numbers after dropping duplicates

In [141]:
# Checking distribution of weighted rating and average rating to quantify benefits of using weighted formula
final_df[['weighted_rating','avg_rating']].describe()

,weighted_rating,avg_rating
count,42329.000000,42329.000000
mean,3.446334,3.064114
std,0.226735,0.717054
min,1.380000,0.500000
25%,3.430000,2.700000
50%,3.500000,3.170000
75%,3.530000,3.500000
max,4.430000,5.000000


#### Observation
Comparing the standard deviations of the two ratings we see
| Metric          | Std Dev |
| --------------- | ------- |
| avg_rating      | 0.717   |
| weighted_rating | 0.226   |

Thus we can conclude that Weighted ratings reduced rating volatility and mitigated the impact of low-review outliers, producing more stable movie quality estimates.


### Exporting the two Cleaned and Feature enriched datasets for further analysis in separate notebook

In [149]:
final_df.to_csv('final_dataset.csv',index=False)
financial_df.to_csv('financial_dataset.csv',index=False)